In [10]:
import re
import numpy as np
import pandas as pd
import itertools
from tqdm.auto import tqdm

import matplotlib.pyplot as plt


from bert_score import score
import pymorphy3
from rapidfuzz.distance import Levenshtein
from sacrebleu.metrics import BLEU, CHRF


import warnings
import logging
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")

hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bert_score").setLevel(logging.ERROR)

In [11]:
files = {
    "11": "c2_augmented_11_llama3.csv",
    "c1": "c2_from_c1_augmented_llama3.csv",
}

# Вспомогательные функции подсчета метрик

In [12]:
morph = pymorphy3.MorphAnalyzer()

def tokenize_words(text):
    text = str(text).lower()
    return re.findall(r"[а-яёa-z]+", text)


def lemmatize_text(text):
    words = tokenize_words(text)
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return lemmas

In [13]:
def bertscore_pair(original_text, augmented_text, model_type="DeepPavlov/rubert-base-cased"):
    """
    Считает BERTScore между двумя текстами:
    original_text — исходный текст
    augmented_text — сгенерированный / аугментированный текст
    """

    P, R, F1 = score(
        [augmented_text],      # candidate
        [original_text],       # reference
        model_type=model_type,
        num_layers=12,
        lang="ru",
        verbose=False
    )

    return {
        "precision": P.item(),
        "recall": R.item(),
        "f1": F1.item()
    }

In [14]:
bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF()


def jaccard_similarity_lemmas(original, augmented):
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas and not aug_lemmas:
        return 1.0

    if not orig_lemmas or not aug_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas | aug_lemmas)


def common_words_ratio(original, augmented):
    """
    Доля лемм исходного текста, которые сохранились в аугментированном.
    """
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas)


def normalized_levenshtein_distance(original, augmented):
    """
    0 — тексты одинаковые.
    1 — тексты максимально различаются.
    """
    original = str(original)
    augmented = str(augmented)

    max_len = max(len(original), len(augmented))

    if max_len == 0:
        return 0.0

    distance = Levenshtein.distance(original, augmented)
    return distance / max_len


def normalized_levenshtein_similarity(original, augmented):
    """
    1 — тексты одинаковые.
    0 — тексты максимально различаются.
    """
    return 1 - normalized_levenshtein_distance(original, augmented)

In [15]:
def lcs_length(x, y):
    """
    Longest Common Subsequence для двух списков токенов.
    """
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_f1(original, augmented):
    orig_tokens = tokenize_words(original)
    aug_tokens = tokenize_words(augmented)

    if not orig_tokens or not aug_tokens:
        return 0.0

    lcs = lcs_length(orig_tokens, aug_tokens)

    precision = lcs / len(aug_tokens)
    recall = lcs / len(orig_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

In [16]:
def bleu_score(original, augmented):
    """
    BLEU: насколько n-граммы аугментированного текста совпадают с исходным.
    Значение приводим к диапазону 0–1.
    """
    score = bleu_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100


def chrf_score(original, augmented):
    """
    chrF: сходство на уровне символьных n-грамм.
    Значение приводим к диапазону 0–1.
    """
    score = chrf_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100

# Сравнение исходных и аугментированных текстов внутри одной температуры

In [17]:
def calculate_pair_metrics(original, augmented):
    return {
        "bert_score": bertscore_pair(original, augmented),
        "jaccard_lemmas": jaccard_similarity_lemmas(original, augmented),
        "common_words_ratio": common_words_ratio(original, augmented),
        "levenshtein_distance": normalized_levenshtein_distance(original, augmented),
        "levenshtein_similarity": normalized_levenshtein_similarity(original, augmented),
        "rouge_l": rouge_l_f1(original, augmented),
        "bleu": bleu_score(original, augmented),
        "chrf": chrf_score(original, augmented),
    }

In [18]:
all_rows = []

for augmentation_type, path in files.items():
    df = pd.read_csv(path)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Обработка строк"):
        original = row['text']
        augmented = row['augmented-text']

        metrics = calculate_pair_metrics(original, augmented)

        metrics["augmentation_type"] = augmentation_type
        metrics["original"] = original
        metrics["augmented"] = augmented

        all_rows.append(metrics)

pair_metrics_df = pd.DataFrame(all_rows)

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

In [19]:
pair_metrics_df['bert_score_precision'] = [pair_metrics_df['bert_score'][i]['precision'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_recall'] = [pair_metrics_df['bert_score'][i]['recall'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_f1'] = [pair_metrics_df['bert_score'][i]['f1'] for i in range(len(pair_metrics_df))]

del pair_metrics_df['bert_score']

pair_metrics_df

,jaccard_lemmas,common_words_ratio,levenshtein_distance,levenshtein_similarity,rouge_l,bleu,chrf,augmentation_type,original,augmented,bert_score_precision,bert_score_recall,bert_score_f1
0,0.633333,0.791667,0.272189,0.727811,0.775510,0.576076,0.802889,11,"3:38–3:54 Собственно, сама по себе радиация не...","Самопо себе радиация не заразна. Например, мощ...",0.808277,0.713196,0.757766
1,0.612903,0.730769,0.294931,0.705069,0.603774,0.272703,0.728419,11,Недавно компания Uber объявила об инвестиции о...,Компания Uber объявила о вложении в миллиард д...,0.857345,0.855500,0.856422
2,0.193548,0.300000,0.816456,0.183544,0.162162,0.159723,0.368095,11,"Множество повестей: «Двойник», «Дядюшкин сон»,...","Многообразие литературных жанров: романы, пове...",0.647117,0.565309,0.603453
3,0.195652,0.375000,0.576471,0.423529,0.250000,0.037346,0.363947,11,Встречи одноклассников и одногруппников превра...,Встречи с одноклассниками и одногруппниками тр...,0.567168,0.555684,0.561368
4,0.384615,0.555556,0.381743,0.618257,0.542373,0.305793,0.574807,11,Самопрезентация — как главная черта времени и ...,Самопрезентация - это доминирующая характерист...,0.806323,0.813633,0.809962
...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,0.307692,0.470588,0.731544,0.268456,0.294118,0.092264,0.421087,c1,"Более того, опрос ВЦИОМа показал, что правосла...","Согласно результатам опроса ВЦИОМа, обнаружило...",0.709846,0.716343,0.713079
236,0.550000,0.880000,0.454839,0.545161,0.600000,0.268165,0.681500,c1,В России традиционными героями новогодней ёлки...,В России традиционные герои новогодней ёлки пр...,0.714652,0.852007,0.777308
237,0.229167,0.611111,0.752000,0.248000,0.238806,0.160927,0.434236,c1,Любой квалифицированный сотрудник сейчас имеет...,Квалифицированные сотрудники сегодня имеют зар...,0.496937,0.658613,0.566465
238,0.235294,0.444444,0.721925,0.278075,0.130435,0.030162,0.365870,c1,"Мне кажется, что вот это москвичам удавалось л...","Я считаю, что Москва традиционно была более ус...",0.470129,0.504197,0.486568


In [20]:
metric_columns = [
    "bert_score_precision",
    "bert_score_recall",
    "bert_score_f1",
    "jaccard_lemmas",
    "common_words_ratio",
    "levenshtein_distance",
    "levenshtein_similarity",
    "rouge_l",
    "bleu",
    "chrf",
]

results_df = (
    pair_metrics_df
    .groupby("augmentation_type")[metric_columns]
    .agg(["mean", "std", "min", "max"])
)

In [21]:
results_df_flat = results_df.copy()

results_df_flat.columns = [
    f"{metric}_{stat}"
    for metric, stat in results_df_flat.columns
]

results_df_flat = results_df_flat.reset_index()

In [22]:
results_df['bert_score_precision']

,mean,std,min,max
augmentation_type,,,,
11,0.806455,0.094969,0.526125,1.000000
c1,0.582762,0.093141,0.348493,0.843956


In [23]:
results_df['bert_score_recall']

,mean,std,min,max
augmentation_type,,,,
11,0.798955,0.098581,0.531702,1.000000
c1,0.663552,0.107773,0.367747,0.908425


In [24]:
results_df['bert_score_f1']

,mean,std,min,max
augmentation_type,,,,
11,0.802238,0.095238,0.528899,1.000000
c1,0.619406,0.096170,0.357861,0.860714


In [25]:
results_df['jaccard_lemmas']

,mean,std,min,max
augmentation_type,,,,
11,0.566846,0.173536,0.193548,1.000000
c1,0.334154,0.113431,0.105263,0.617647


In [26]:
results_df['common_words_ratio']

,mean,std,min,max
augmentation_type,,,,
11,0.711777,0.145703,0.30000,1.0
c1,0.634810,0.157935,0.26087,1.0


In [27]:
results_df['levenshtein_distance']

,mean,std,min,max
augmentation_type,,,,
11,0.347500,0.167273,0.000000,0.816456
c1,0.613415,0.105968,0.352113,0.810651


In [28]:
results_df['levenshtein_similarity']

,mean,std,min,max
augmentation_type,,,,
11,0.652500,0.167273,0.183544,1.000000
c1,0.386585,0.105968,0.189349,0.647887


In [29]:
results_df['rouge_l']

,mean,std,min,max
augmentation_type,,,,
11,0.623235,0.177904,0.162162,1.000000
c1,0.361858,0.131780,0.092308,0.698413


In [30]:
results_df['bleu']

,mean,std,min,max
augmentation_type,,,,
11,0.388534,0.221376,0.027033,1.000000
c1,0.140873,0.096880,0.010752,0.438305


In [31]:
results_df['chrf']

,mean,std,min,max
augmentation_type,,,,
11,0.653674,0.146361,0.302299,1.000000
c1,0.503472,0.114600,0.270381,0.791328
